# Feature Extraction: Abnormal Sentiment

This notebook implements **Abnormal Sentiment** features.

**Definition:**
Abnormal Sentiment is defined as the deviation of the current day's sentiment from a moving average of past sentiment.
$$ \text{Abnormal Sentiment}_{t, H} = \text{Sentiment}_t - \text{MovingAverage}(\text{Sentiment}, H) $$

**Horizons ($H$):**
We calculate this for multiple horizons: $H \in \{1, 5, 21, 63, 250\}$ trading days.
*   $H=1$: Daily change ($\Delta S_t$)
*   $H>1$: Deviation from trend

**Methodology:**
1.  Load `features_01_basic_sentiment.pkl`.
2.  Create a **Complete Trading Day Grid** (Symbol x Date) to ensure rolling windows respect calendar time (trading days).
3.  **Imputation:** Days with no tweets are assigned a **Net Sentiment of 0 (Neutral)**. This ensures that periods of silence contribute to the moving average as neutral sentiment, rather than being skipped.
4.  Calculate rolling means and deviations.
5.  Save to `features_03_abnormal_sentiment.pkl`.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import pandas_datareader as pdr

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# =============================================================================
# DIRECTORY PATHS
# =============================================================================
DATA_DIR = Path(r"c:\Users\skazempour\Documents\StockTwits\dataset\v1\data\csv")
INPUT_FILE = DATA_DIR / "features_mlcrowd" / "features_01_basic_sentiment.pkl"
OUTPUT_FOLDER = DATA_DIR / "features_mlcrowd"
OUTPUT_FILE = OUTPUT_FOLDER / "features_03_abnormal_sentiment.pkl"

print(f"Input file: {INPUT_FILE}")
print(f"Output file: {OUTPUT_FILE}")

c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Input file: c:\Users\skazempour\Documents\StockTwits\dataset\v1\data\csv\features_mlcrowd\features_01_basic_sentiment.pkl
Output file: c:\Users\skazempour\Documents\StockTwits\dataset\v1\data\csv\features_mlcrowd\features_03_abnormal_sentiment.pkl


## 1. Load Basic Sentiment Features

## 1. Load Trading Day Calendar from Fama-French Data
We use the Fama-French daily factors to get the official trading day calendar. This ensures we have all CRSP trading days, even if no tweets occurred on a specific day.

In [3]:
print("Loading Fama-French data for trading day calendar...")
try:
    # Download FF3 daily factors (has all trading days)
    # We start from 2009 to cover the likely range of our data
    ff_data = pdr.data.DataReader('F-F_Research_Data_Factors_daily', 'famafrench', start='2009')[0]
    
    # Extract trading days
    ALL_TRADING_DAYS = pd.to_datetime(ff_data.index)
    print(f"✓ Loaded {len(ALL_TRADING_DAYS):,} trading days from Fama-French")
    print(f"  Date range: {ALL_TRADING_DAYS.min().date()} to {ALL_TRADING_DAYS.max().date()}")
    
except Exception as e:
    print(f"Warning: Could not download FF data: {e}")
    print("Will extract trading days from CRSP-merged data instead")
    ALL_TRADING_DAYS = None

Loading Fama-French data for trading day calendar...


C:\Users\skazempour\AppData\Local\Temp\ipykernel_22308\2441857060.py:5: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ff_data = pdr.data.DataReader('F-F_Research_Data_Factors_daily', 'famafrench', start='2009')[0]


✓ Loaded 4,254 trading days from Fama-French
  Date range: 2009-01-02 to 2025-11-28


## 2. Load Basic Sentiment Features

In [4]:
# Load data
if INPUT_FILE.exists():
    df = pd.read_pickle(INPUT_FILE)
    print(f"Loaded data shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    
    # Ensure date is datetime
    df['date'] = pd.to_datetime(df['date'])
else:
    print(f"Error: Input file not found at {INPUT_FILE}")
    # Stop execution if file missing (in a real script, we'd raise an error)

Loaded data shape: (3505817, 13)
Columns: ['symbol', 'date', 'n_bullish', 'n_bearish', 'total_labeled', 'bullish_ratio', 'bearish_ratio', 'net_sentiment', 'extreme_bullish_80', 'extreme_bullish_90', 'extreme_bearish_80', 'extreme_bearish_90', 'disagreement_index']


## 3. Create Complete Trading Day Grid
To ensure rolling windows are accurate (e.g., "5 days" means 5 trading days, not just 5 observations which might span weeks if data is sparse), we create a complete grid of `(Symbol, Date)` for every trading day in the range.

In [5]:
# 1. Identify all trading days
if ALL_TRADING_DAYS is not None:
    # Filter to date range in the data
    data_start = df['date'].min()
    data_end = df['date'].max()
    all_trading_days = ALL_TRADING_DAYS[(ALL_TRADING_DAYS >= data_start) & (ALL_TRADING_DAYS <= data_end)]
    print(f"Using Fama-French calendar: {len(all_trading_days):,} trading days")
else:
    # Fallback: extract from data
    all_trading_days = sorted(df['date'].unique())
    print(f"Using data-derived calendar: {len(all_trading_days):,} days")

print(f"Date range: {all_trading_days[0].date()} to {all_trading_days[-1].date()}")

# 2. Identify all symbols
all_symbols = df['symbol'].unique()
print(f"Total symbols: {len(all_symbols):,}")

# 3. Create Cartesian Product (Grid)
print("Creating complete grid...")
complete_grid = pd.MultiIndex.from_product(
    [all_symbols, all_trading_days],
    names=['symbol', 'date']
).to_frame(index=False)

print(f"Grid size: {len(complete_grid):,} rows")

# 4. Merge Data
print("Merging sentiment data into grid...")
df_complete = complete_grid.merge(df, on=['symbol', 'date'], how='left')

# 5. Handle Missing Values
# - Counts: Fill with 0 (no tweets = 0 counts)
# - Sentiment: Fill with 0 (Neutral) for days with no tweets
#   This is crucial for Abnormal Sentiment: silence is treated as neutral, 
#   so a sudden spike in sentiment is abnormal relative to silence.

count_cols = ['n_bullish', 'n_bearish', 'total_labeled', 
              'extreme_bullish_80', 'extreme_bullish_90',
              'extreme_bearish_80', 'extreme_bearish_90']
df_complete[count_cols] = df_complete[count_cols].fillna(0)

# Fill Net Sentiment with 0 (Neutral)
df_complete['net_sentiment'] = df_complete['net_sentiment'].fillna(0)

# Note: Disagreement Index is not used for Abnormal Sentiment calculations,
# so we do not need to impute it here. It will remain NaN for missing days.

print(f"Complete dataset shape: {df_complete.shape}")
print(f"Missing sentiment values after fill: {df_complete['net_sentiment'].isnull().sum():,}")

Using Fama-French calendar: 3,421 trading days
Date range: 2010-06-02 to 2024-01-03
Total symbols: 8,245
Creating complete grid...
Grid size: 28,206,145 rows
Merging sentiment data into grid...
Complete dataset shape: (28206145, 13)
Missing sentiment values after fill: 0


## 4. Calculate Abnormal Sentiment
We calculate the deviation from the moving average for various horizons.
$$ \text{Abnormal}_{t, H} = S_t - \text{Mean}(S_{t-H+1} \dots S_t) $$

In [6]:
# Sort for rolling calculations
df_complete = df_complete.sort_values(['symbol', 'date'])

horizons = [1, 5, 21, 63, 250]
print(f"Calculating Abnormal Sentiment for horizons: {horizons}")

# Group by symbol
grouped = df_complete.groupby('symbol')['net_sentiment']

for h in tqdm(horizons, desc="Horizons"):
    col_name = f'abnormal_sentiment_{h}d'
    
    if h == 1:
        # Daily change (Delta)
        df_complete[col_name] = grouped.diff(1)
    else:
        # Deviation from Moving Average
        # min_periods=1 allows calculation even if some days in window are missing (NaN)
        # However, if S_t is NaN, the result will be NaN, which is correct.
        rolling_mean = grouped.rolling(window=h, min_periods=1).mean().reset_index(0, drop=True)
        df_complete[col_name] = df_complete['net_sentiment'] - rolling_mean

# Inspect results
new_cols = [f'abnormal_sentiment_{h}d' for h in horizons]
print("\nSample results:")
display(df_complete[['symbol', 'date', 'net_sentiment'] + new_cols].head(10))

print("\nSummary Statistics:")
display(df_complete[new_cols].describe())

Calculating Abnormal Sentiment for horizons: [1, 5, 21, 63, 250]


Horizons:   0%|          | 0/5 [00:00<?, ?it/s]


Sample results:


,symbol,date,net_sentiment,abnormal_sentiment_1d,abnormal_sentiment_5d,abnormal_sentiment_21d,abnormal_sentiment_63d,abnormal_sentiment_250d
0,A,2010-06-02,0.0,NaN,0.0,0.0,0.0,0.0
1,A,2010-06-03,0.0,0.0,0.0,0.0,0.0,0.0
2,A,2010-06-04,0.0,0.0,0.0,0.0,0.0,0.0
3,A,2010-06-07,0.0,0.0,0.0,0.0,0.0,0.0
4,A,2010-06-08,0.0,0.0,0.0,0.0,0.0,0.0
5,A,2010-06-09,0.0,0.0,0.0,0.0,0.0,0.0
6,A,2010-06-10,0.0,0.0,0.0,0.0,0.0,0.0
7,A,2010-06-11,0.0,0.0,0.0,0.0,0.0,0.0
8,A,2010-06-14,0.0,0.0,0.0,0.0,0.0,0.0
9,A,2010-06-15,0.0,0.0,0.0,0.0,0.0,0.0



Summary Statistics:


,abnormal_sentiment_1d,abnormal_sentiment_5d,abnormal_sentiment_21d,abnormal_sentiment_63d,abnormal_sentiment_250d
count,2.819790e+07,2.820614e+07,2.820614e+07,2.820614e+07,2.820614e+07
mean,4.290857e-06,2.687402e-05,1.064154e-04,1.252645e-04,2.183566e-03
std,3.219291e-01,2.094844e-01,2.367217e-01,2.476787e-01,2.617572e-01
min,-2.000000e+00,-1.600000e+00,-1.904762e+00,-1.952786e+00,-1.965273e+00
25%,0.000000e+00,0.000000e+00,0.000000e+00,-1.587302e-02,-2.000000e-02
50%,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
max,2.000000e+00,1.600000e+00,1.814744e+00,1.788912e+00,1.607335e+00


## 5. Save Results
We save the complete dataset (including the filled trading days) to a new pickle file. This ensures downstream tasks have a continuous time series.

In [7]:
# Save features to pickle
print(f"Saving features to: {OUTPUT_FILE}")
df_complete.to_pickle(OUTPUT_FILE)
print(f"✓ Saved successfully!")

# Verify save
print(f"\nVerifying saved file...")
features_loaded = pd.read_pickle(OUTPUT_FILE)
print(f"✓ File readable")
print(f"✓ Shape matches: {features_loaded.shape == df_complete.shape}")
print(f"✓ Columns match: {list(features_loaded.columns) == list(df_complete.columns)}")

# File size
file_size_mb = OUTPUT_FILE.stat().st_size / 1024**2
print(f"\nFile size: {file_size_mb:.2f} MB")

Saving features to: c:\Users\skazempour\Documents\StockTwits\dataset\v1\data\csv\features_mlcrowd\features_03_abnormal_sentiment.pkl
✓ Saved successfully!

Verifying saved file...
✓ File readable
✓ Shape matches: True
✓ Columns match: True

File size: 4005.91 MB


## Summary

✓ **Features Implemented:**
- **Feature 11:** Abnormal Sentiment (Horizons: 1, 5, 21, 63, 250 days)

✓ **Data Improvements:**
- Created a **Complete Trading Day Grid** for all symbols.
- Merged sentiment data, filling missing days with appropriate values (0 for counts, NaN for sentiment).
- Ensures rolling window calculations respect calendar time.

✓ **Output:** `features_03_abnormal_sentiment.pkl`